# 12 — Model\n**ENAI 603 | WMATA Metro Delay Prediction**\n\nModels trained in this notebook:\n1. \n\nEvaluation: AUC-ROC, precision, recall, F1. Time-series aware split (train on earlier data, test on later).

In [3]:
#pip install catboost scikit-learn pandas

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import folium

from folium.plugins import HeatMap
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay, f1_score, precision_score, recall_score
)

sns.set_theme(style="whitegrid", font_scale=1.1)

PROJ = os.path.abspath(os.path.join(os.getcwd(), ".."))
FIG_DIR = os.path.join(PROJ, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROJ, "data", "features.csv"), parse_dates=["collected_at"])
df = df[df["is_orphan"] == 0]
print(f"Loaded and orphans filtered... {len(df):,} rows")

Loaded 11,175,644 rows


## 1. Prepare Features & Time-Series Split

In [5]:
# Define feature groups
NUMERIC_FEATURES = [
    "hour", "day_of_week", "is_weekend", "is_rush_hour",
    "num_lines", "is_terminal", "lat", "lon",
    "minutes_num", "num_predictions_in_feed", "avg_cars_at_station",
    "delay_rate_30min", "line_delay_rate_30min",
    "active_incident", "incident_is_delay",
    "scheduled_headway_min", "loc_has_parking",
    "num_rails_conn"
]

CATEGORICAL_FEATURES = ["line"]

TARGET = "is_delayed"

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
df_model = df[feature_cols + [TARGET, "collected_at"]].copy()
# Fill NaN in numeric features with median
for col in NUMERIC_FEATURES:
    if df_model[col].isnull().any():
        df_model[col] = df_model[col].fillna(df_model[col].median())

# Drop rows with NaN in features we'll use
df_model = df_model.dropna(subset=[TARGET])

# Time-series split: train on first 80% chronologically, test on last 20%
df_model = df_model.sort_values("collected_at").reset_index(drop=True)
split_idx = int(len(df_model) * 0.8)

train = df_model.iloc[:split_idx]
test = df_model.iloc[split_idx:]

X_train = train[feature_cols]
y_train = train[TARGET]
X_test = test[feature_cols]
y_test = test[TARGET]

print(f"Train: {len(train):,} rows ({train['collected_at'].min()} → {train['collected_at'].max()})")
print(f"Test:  {len(test):,} rows ({test['collected_at'].min()} → {test['collected_at'].max()})")
print(f"\nTrain delay rate: {y_train.mean():.2%}")
print(f"Test delay rate:  {y_test.mean():.2%}")

Train: 8,940,515 rows (2026-03-11 02:00:05.155939+00:00 → 2026-04-15 18:44:00.070410+00:00)
Test:  2,235,129 rows (2026-04-15 18:44:00.070410+00:00 → 2026-04-28 22:24:04.713007+00:00)

Train delay rate: 23.51%
Test delay rate:  22.52%


In [6]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

## 2. Model Training & Evaluation

In [7]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, XGB=False):
    """Train, predict, and return metrics dict."""
    if XGB:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_te, y_te)], early_stopping_rounds=100,
            verbose=200)
    else:
        model.fit(X_tr, y_tr)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        y_prob = model.predict(X_te).astype(float)

    y_pred = model.predict(X_te)
    auc = roc_auc_score(y_te, y_prob)

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  Precision: {precision_score(y_te, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_te, y_pred):.4f}")
    print(f"  F1:        {f1_score(y_te, y_pred):.4f}")
    print()
    print(classification_report(y_te, y_pred, target_names=["On-time", "Delayed"]))

    return {"name": name, "model": model, "auc": auc, "y_prob": y_prob, "y_pred": y_pred}


results = []

# 1. Majority-class baseline
dummy = DummyClassifier(strategy="most_frequent")
results.append(evaluate_model("Majority Baseline", dummy, X_train, y_train, X_test, y_test))

# 2. Logistic Regression
lr_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
results.append(evaluate_model("Logistic Regression", lr_pipe, X_train, y_train, X_test, y_test))

# 3. Random Forest
rf_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=20,
        class_weight="balanced", random_state=42, n_jobs=-1
    )),
])
results.append(evaluate_model("Random Forest", rf_pipe, X_train, y_train, X_test, y_test))


  Majority Baseline
  AUC-ROC:   0.5000


C:\Users\ftruj\WMATA-Metro-Delay-Prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  Precision: 0.0000
  Recall:    0.0000
  F1:        0.0000



C:\Users\ftruj\WMATA-Metro-Delay-Prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ftruj\WMATA-Metro-Delay-Prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ftruj\WMATA-Metro-Delay-Prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

              precision    recall  f1-score   support

     On-time       0.77      1.00      0.87   1731756
     Delayed       0.00      0.00      0.00    503373

    accuracy                           0.77   2235129
   macro avg       0.39      0.50      0.44   2235129
weighted avg       0.60      0.77      0.68   2235129


  Logistic Regression
  AUC-ROC:   0.7318
  Precision: 0.3649
  Recall:    0.6481
  F1:        0.4669

              precision    recall  f1-score   support

     On-time       0.87      0.67      0.76   1731756
     Delayed       0.36      0.65      0.47    503373

    accuracy                           0.67   2235129
   macro avg       0.62      0.66      0.61   2235129
weighted avg       0.75      0.67      0.69   2235129


  Random Forest
  AUC-ROC:   0.7503
  Precision: 0.3389
  Recall:    0.8287
  F1:        0.4810

              precision    recall  f1-score   support

     On-time       0.91      0.53      0.67   1731756
     Delayed       0.34      0.83  

In [ ]:
# 4. XGBoost
xgb_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=500,#2000,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        tree_method="hist",
        enable_categorical=True)
     )
])

results.append(evaluate_model("XGBoost Classifier", xgb_pipe, X_train, y_train, X_test, y_test, XGB=False))

In [ ]:
# 5. CatBoost Classifier
# Try different iterations to see variation
num_iterations = [1000]#, 2000]
for iter in num_iterations:
    catbc_pipe = Pipeline([
        ("pre", preprocessor),
        ("clf", CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=iter,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            random_seed=42,
            verbose=200,
            early_stopping_rounds=100)
         )
    ])
    results.append(evaluate_model(f"CatBoost Classifier (iter:{iter})",
                                  catbc_pipe ,
                                  X_train , y_train,
                                  X_test, y_test))

## 3. ROC Curves Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
colors = ["gray", "#1976D2", "#388E3C", "yellow", "orange", "black"]
for res, color in zip(results, colors):
    RocCurveDisplay.from_predictions(
        y_test, res["y_prob"],
        name=f"{res['name']} (AUC={res['auc']:.3f})",
        ax=axes[0], color=color
    )
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title("ROC Curves")

# AUC comparison bar chart
names = [r["name"] for r in results]
aucs = [r["auc"] for r in results]
bar_colors = colors[:len(results)]
axes[1].barh(names, aucs, color=bar_colors)
axes[1].set_xlabel("AUC-ROC")
axes[1].set_title("Model Comparison")
axes[1].set_xlim(0.4, 1.0)
axes[1].axvline(0.80, color="red", ls="--", alpha=0.5, label="Target (0.80)")
axes[1].legend()
for i, v in enumerate(aucs):
    axes[1].text(v + 0.01, i, f"{v:.3f}", va="center", fontweight="bold")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "baseline_roc_curves.png"), dpi=150)
plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.ravel()

for ax, res in zip(axes, results):
    cm = confusion_matrix(y_test, res["y_pred"])
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
                xticklabels=["On-time", "Delayed"],
                yticklabels=["On-time", "Delayed"])
    ax.set_title(f"{res['name']}\nAUC={res['auc']:.3f}")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "baseline_confusion_matrices.png"), dpi=150)
plt.show()

## 5. Feature Importance

In [ ]:
def features_importance(model) -> pd.Series:
    if(isinstance(model, DummyClassifier)):
        return pd.Series(dtype=float)

    if hasattr(model, "named_steps"):
        ohe_features = list(model.named_steps["pre"]
                            .get_feature_names_out())
    else:
        ohe_features = list(model.get_feature_names_out())

    clf = model.named_steps["clf"]
    if hasattr(clf, "feature_importances_"):
        print("Feature importance")
        importance = clf.feature_importances_
    elif hasattr(clf, "coef_"):
        print("Coefficient")
        importance = np.abs(clf.coef_).ravel()
    else:
        return pd.Series(dtype=float)
    return pd.Series(importance, index=ohe_features).sort_values()

results_feature_importance = []

for result in results:
    print(f"Model: {result['name']}")
    results_feature_importance.append(
        (result['name'], features_importance(result['model'])))

# List of results to plot
r_to_plot = ["Logistic Regression", "Random Forest", "XGBoost Classifier", "CatBoost Classifier (iter:1000)"]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
j = 0
for name_mod, series_feat in results_feature_importance:
    if (not(series_feat.empty) and (name_mod in r_to_plot)):
        series_feat.tail(15).plot(kind="barh", ax=axes[int(j/2)][j%2])
        axes[int(j/2)][j%2].set_xlabel("Feature Importance")
        axes[int(j/2)][j%2].set_title(f"Top 15 Features — {name_mod}")
        j += 1

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Feature_importance_models.png"), dpi=150)
plt.show()

## 6. Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        "Model": r["name"],
        "AUC-ROC": r["auc"],
        "Precision": precision_score(y_test, r["y_pred"]),
        "Recall": recall_score(y_test, r["y_pred"]),
        "F1": f1_score(y_test, r["y_pred"]),
    }
    for r in results
])
summary = summary.round(4)
print(summary.to_string(index=False))